# Transformación de contratos electrónicos de SECOP II

Este notebook utiliza PySpark para transformar los contratos electrónicos de SECOP II correspondientes a septiembre de 2026. Se aplican reglas de calidad y organización para preparar los datos para su posterior análisis, conservando los registros originales de MongoDB.


## 1. Configuración

Se define el periodo de análisis, el territorio y los parámetros necesarios para ejecutar la transformación. La escritura en PostgreSQL permanece desactivada mientras se validan los resultados.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

NB_DIR = Path("/home/jovyan/work/spark-apps/notebooks")
if not NB_DIR.exists():
    NB_DIR = Path.cwd().resolve()
    if NB_DIR.name == "transformation":
        NB_DIR = NB_DIR.parent
SRC = NB_DIR.parent / "src"
sys.path.insert(0, str(SRC))

from transformation.spark_sesion import _asegurar_pyspark, crear_spark

_asegurar_pyspark()
from pyspark.sql import functions as F
from transformation.controles import CONTROLES_CRITICOS_CONTRATOS, controles_criticos_ok
from transformation.contratos import invariantes_contrato
from transformation.io import persistir_periodo_contratos, verificar_persistencia
from transformation.jobs.job_contratos import ejecutar

PERIODO = "2026-09"
ESCRIBIR = False  # True solo tras validar conteos
TERRITORIO = "nacional"

SALIDAS = NB_DIR / "salidas" / "transformation" / PERIODO / "contratos"
FIGURAS = SALIDAS / "figuras"
SALIDAS.mkdir(parents=True, exist_ok=True)
FIGURAS.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.titlesize"] = 11

print("Periodo:", PERIODO)
print("ESCRIBIR:", ESCRIBIR)
print("Territorio:", TERRITORIO)
print("Salidas:", SALIDAS)
print("MONGO_URI definido:", bool(os.environ.get("MONGO_URI")))
print("PGUSER definido:", bool(os.environ.get("PGUSER")))


## 2. Spark local y lectura

Se inicia la sesión de Spark para realizar las operaciones de transformación y procesamiento de los contratos electrónicos.

In [ ]:
spark = crear_spark("nb07-transformacion-contratos")
print("Spark:", spark.version, spark.sparkContext.master)


## 3. Transformación (mismas funciones que el job)

Se aplican las reglas de transformación para obtener un registro por contrato, verificar sus identificadores, utilizar la fecha de firma y marcar valores que requieren revisión. Los contratos se procesan de manera independiente de los procesos de contratación.

In [ ]:
res = ejecutar(spark, periodo=PERIODO, escribir=False)
filas = res["filas"]
contrato = res["contrato"]
cuadro = pd.DataFrame(res["cuadro"])
print("Contratos derivados:", contrato.count())
print("Invariantes:", res["fallos_invariante"] or "ok")
display(cuadro)
cuadro.to_csv(SALIDAS / "control_reconciliacion.csv", index=False)


## 4. Territorio y calidad

Se revisa la información territorial de los contratos y se identifican los registros con valores en cero, negativos o extremos. También se permite seleccionar únicamente los contratos correspondientes a Bogotá.

In [ ]:
if TERRITORIO == "bogota":
    vista = contrato.filter(F.col("es_bogota"))
    print("Recorte Bogotá:", vista.count())
else:
    vista = contrato
    print("Sin recorte territorial (nacional).")

hom = (vista.groupBy("estado_homologacion").count()
       .orderBy(F.col("count").desc()).toPandas())
display(hom)
hom.to_csv(SALIDAS / "homologacion.csv", index=False)

print("Ceros / negativos / extremos:",
      res["observado"]["contratos_valor_cero"],
      res["observado"]["contratos_valor_negativo"],
      res["observado"]["contratos_valor_extremo"])


## 5. Análisis de los valores de los contratos

Se analiza la distribución de los valores de los contratos mediante estadísticas y gráficos. Se identifican los valores aptos para el análisis de anomalías, conservando por separado los registros con valores en cero, nulos o extremos.

In [ ]:
apto = vista.filter(F.col("valor_apto_modelo")).select("valor_del_contrato")
qs = [0.5, 0.95, 0.99]
qs_vals = apto.approxQuantile("valor_del_contrato", qs, 0.001)
p = {f"p{int(q*100)}": r for q, r in zip(qs, qs_vals)}
n_apto = apto.count()
suma = apto.agg(F.sum("valor_del_contrato").alias("s")).collect()[0]["s"]
p99 = p["p99"]
suma_top = (apto.filter(F.col("valor_del_contrato") >= F.lit(p99))
            .agg(F.sum("valor_del_contrato").alias("s")).collect()[0]["s"])
print("n_apto_modelo:", n_apto, "de", vista.count())
print("percentiles aptos:", p)
print("share suma >= P99:", None if not suma else round(float(suma_top) / float(suma), 4))

ext = (vista.filter(F.col("valor_del_contrato_extremo"))
       .select("id_contrato", "nombre_entidad", "departamento",
               "modalidad_de_contratacion", "valor_del_contrato")
       .toPandas())
print("Extremos (> 1e12):")
display(ext)
ext.to_csv(SALIDAS / "valores_extremos.csv", index=False)

pdf = (apto.select(F.log10(F.col("valor_del_contrato")).alias("log10_valor"))
       .toPandas())
fig, ax = plt.subplots()
ax.hist(pdf["log10_valor"], bins=40, color="#4C78A8")
ax.set_title("valor_del_contrato apto (log10)")
ax.set_xlabel("log10(valor)")
plt.tight_layout()
plt.savefig(FIGURAS / "valor_log10.png", dpi=120, bbox_inches="tight")
plt.show()

pd.DataFrame([{
    "n_contratos": int(vista.count()),
    "n_apto_modelo": int(n_apto),
    "n_cero": res["observado"]["contratos_valor_cero"],
    "n_negativo": res["observado"]["contratos_valor_negativo"],
    "n_extremo": res["observado"]["contratos_valor_extremo"],
    **p,
}]).to_csv(SALIDAS / "valor_cobertura.csv", index=False)


## 6. Distribución por departamentos y modalidades

Se analiza la cantidad de contratos por departamento y modalidad de contratación. Los resultados se presentan mediante tablas y gráficos para facilitar su comparación.

In [ ]:
dep = (vista.groupBy("departamento").count().orderBy(F.col("count").desc())
       .limit(12).toPandas())
mod = (vista.groupBy("modalidad_de_contratacion").count()
       .orderBy(F.col("count").desc()).limit(8).toPandas())
display(dep)
display(mod)
dep.to_csv(SALIDAS / "departamentos.csv", index=False)
mod.to_csv(SALIDAS / "modalidades.csv", index=False)

fig, ax = plt.subplots()
dep.set_index("departamento")["count"].plot(kind="barh", ax=ax, color="#4C78A8")
ax.invert_yaxis()
ax.set_title("Contratos por departamento de la entidad")
plt.tight_layout()
plt.savefig(FIGURAS / "departamentos.png", dpi=120, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots()
mod.set_index("modalidad_de_contratacion")["count"].plot(kind="bar", ax=ax, color="#72B7B2")
ax.set_title("Modalidades")
plt.tight_layout()
plt.savefig(FIGURAS / "modalidades.png", dpi=120, bbox_inches="tight")
plt.show()


## 7. Preparación para el almacenamiento

Se define el procedimiento para almacenar los contratos transformados en PostgreSQL, incluyendo las validaciones necesarias antes de publicar la información. En esta configuración, la escritura permanece desactivada y no se modifica la información de los procesos.


In [ ]:
print("ESCRIBIR =", ESCRIBIR)
print("tfm.contrato:", contrato.count(), "filas")
if ESCRIBIR:
    if not controles_criticos_ok(res["cuadro"], CONTROLES_CRITICOS_CONTRATOS):
        print("Control crítico distinto del EDA. No se publica.")
    elif res.get("fallos_invariante"):
        print("Invariantes fallidas. No se publica:", res["fallos_invariante"])
    else:
        persistir_periodo_contratos(contrato, res["control"], PERIODO)
        verif = verificar_persistencia(
            spark, PERIODO, {"contrato": res["observado"]["contratos"]},
            pks={"contrato": ["periodo", "id_contrato"]},
        )
        display(pd.DataFrame(verif))
        print("Escritura tfm.contrato hecha.")
else:
    print("Sin escritura. Para persistir: ESCRIBIR=true en job_contratos.py")
pd.DataFrame([{
    "tabla": "tfm.contrato",
    "filas": res["observado"]["contratos"],
    "grano": "periodo + id_contrato",
}]).to_csv(SALIDAS / "tablas_pendientes_tfm.csv", index=False)


## 8. Cierre


In [ ]:
print("Mongo contratos_electronicos_raw: solo lectura.")
print("tfm.proceso / public: no se tocan.")
spark.stop()
print("SparkSession cerrada.")
